# Hands-on: LLM-as-a-Judge für Softwarequalität

**Hinweis:** Das Setup lädt den Code automatisch. Klappt das nicht, ladet die ZIP [hier von Google Drive](https://drive.google.com/file/d/192TGNh2VbD1e-Qp_WVU7CJn5Ra31uSAs/view?usp=sharing) herunter, ladet sie in Colab links über 📁 Dateien → Hochladen hoch (Name `software-quality-engineering.zip` beibehalten) und führt das Setup noch einmal aus.

In [ ]:
#@title ⚙️ Setup (einmal ausführen)

import os
import subprocess
import sys

REPO_URL = "https://github.com/evabowe/software-quality-engineering"
# Ersatz, solange das GitHub-Repo nicht öffentlich ist: ZIP des Repos auf Google Drive.
ZIP_URL = "https://drive.usercontent.google.com/download?id=192TGNh2VbD1e-Qp_WVU7CJn5Ra31uSAs&export=download&confirm=t"
# Lokal (Jupyter im Repo) wird der vorhandene Checkout genutzt, in Colab ein frischer Klon.
if os.path.isfile("../pyproject.toml"):
    REPO_DIR = os.path.abspath("..")
else:
    REPO_DIR = "/content/software-quality-engineering"
    ZIP_PATH = "/content/software-quality-engineering.zip"
    if not os.path.isdir(REPO_DIR):
        geklont = subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR],
                                 capture_output=True).returncode == 0
        if not geklont:
            # GitHub nicht erreichbar: von Hand hochgeladene ZIP nehmen, sonst die von Google Drive laden.
            import urllib.request
            import zipfile

            if not os.path.isfile(ZIP_PATH):
                urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
            with zipfile.ZipFile(ZIP_PATH) as archiv:
                archiv.extractall("/content")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)  # die Installation wird sonst erst nach einem Neustart sichtbar

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import html

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, display

from judge_lab import bewerte, bewerte_alle, sende, verbinde
from judge_lab.snippets import get_snippet, load_snippets

# --- Hilfen für die Zellen der Runden ---------------------------------------------

SNIPPETS = load_snippets()
TITEL = {s.id: s.title for s in SNIPPETS}


def prompt_feld(platzhalter, hoehe="12em"):
    feld = widgets.Textarea(placeholder=platzhalter, layout=widgets.Layout(width="100%", height=hoehe))
    feld.add_class("judge-prompt")
    return feld


def ohne_rechtschreibpruefung():
    # Textarea hat keine Option dafür: Rechtschreibprüfung des Browsers (Englisch) in den Feldern abschalten.
    display(HTML("""<script>
new MutationObserver(() => document.querySelectorAll(".judge-prompt textarea")
    .forEach(t => t.spellcheck = false)).observe(document.body, {childList: true, subtree: true});
</script>"""))


def snippet_auswahl(beschreibung="Beispiel:"):
    optionen = [(f"{s.title} ({s.level or 'ohne Stufe'})", s.id) for s in SNIPPETS]
    return widgets.Dropdown(options=optionen, value="bank_account", description=beschreibung)


def snippet_html(snippet_id):
    snippet = get_snippet(snippet_id)
    return (f"<h4>{html.escape(snippet.title)} <small>({snippet.level or 'ohne Stufe'})</small></h4>"
            f"<pre style='font-size:13px;line-height:1.35'>{html.escape(snippet.numbered_code())}</pre>")


def ergebnis_html(ergebnis, titel=""):
    kopf = f"<b>{html.escape(titel)}</b><br>" if titel else ""
    if ergebnis["fehler"]:
        return f"{kopf}<span style='color:#b45309'>⚠️ {html.escape(ergebnis['fehler'])} Bitte noch einmal bewerten.</span>"
    return (f"{kopf}🎯 Judge-Score <b>{ergebnis['judge_score']}</b> von max. {ergebnis['judge_max']} · "
            f"📝 Code-Score <b>{ergebnis['code_score']:.1f}</b> von 10<br>"
            f"✔ {ergebnis['gefunden']} von {ergebnis['fehler_gesamt']} echten Fehlern gefunden · "
            f"✘ {ergebnis['fehlalarme']} Fehlalarm(e)")


def ergebnis_tabelle(ergebnisse):
    zeilen = {}
    for e in ergebnisse:
        ok = not e["fehler"]
        zeilen[TITEL.get(e["snippet"], e["snippet"])] = {
            "Stufe": e["komplexitaet"],
            "🎯 Judge-Score": f"{e['judge_score']} / {e['judge_max']}" if ok else "–",
            "📝 Code-Score": f"{e['code_score']:.1f}" if ok else "–",
            "✔ gefunden": f"{e['gefunden']} / {e['fehler_gesamt']}" if ok else "–",
            "✘ Fehlalarme": e["fehlalarme"] if ok else "–",
            "Hinweis": e["fehler"] or "",
        }
    return pd.DataFrame.from_dict(zeilen, orient="index")


def knopf(beschreibung, ausgabe, aktion, stil="primary", aktiv=True, aktiv_danach=lambda: True):
    """Button, der während der Aktion gesperrt ist; Meldungen der Funktionen erscheinen in ``ausgabe``."""
    button = widgets.Button(description=beschreibung, button_style=stil, disabled=not aktiv,
                            layout=widgets.Layout(width="auto"))

    def klick(_):
        button.disabled = True
        ausgabe.clear_output()
        try:
            with ausgabe:
                aktion()
        except Exception as exc:  # noqa: BLE001 – im Workshop lieber Meldung als Traceback
            with ausgabe:
                display(HTML(f"<b>⚠️ Unerwarteter Fehler ({html.escape(type(exc).__name__)}). Bitte Bescheid geben.</b>"))
        finally:
            button.disabled = not aktiv_danach()

    button.on_click(klick)
    return button


# --- Verbinden ----------------------------------------------------------------------

gruppencode = widgets.Text(description="Gruppencode:", placeholder="von der Folie",
                           style={"description_width": "initial"})
verbinden_ausgabe = widgets.Output()
verbinden_knopf = knopf("Verbinden", verbinden_ausgabe, lambda: verbinde(gruppencode.value))
display(widgets.HBox([gruppencode, verbinden_knopf]), verbinden_ausgabe)

## Schritt 0: Die Code-Beispiele ansehen

Bevor ihr einen Judge-Prompt schreibt, schaut euch die sechs Code-Beispiele selbst genau an. Sie liegen im Repository im Ordner `snippets/`, je Beispiel in einem eigenen Unterordner (in Colab links über 📁 Dateien → `software-quality-engineering/snippets/`):

| Beispiel | Datei |
|---|---|
| Bankkonto | `snippets/bank_account/bank_account.py` |
| Bitcount | `snippets/qb_bitcount/bitcount.py` |
| Levenshtein Distance | `snippets/qb_levenshtein/levenshtein.py` |
| Next Permutation | `snippets/qb_next_permutation/next_permutation.py` |
| QuickSort | `snippets/qb_quicksort/quicksort.py` |
| Infix to RPN Conversion | `snippets/qb_shunting_yard/shunting_yard.py` |

**Wichtig:** Schaut euch nur die Code-Dateien aus der Tabelle an – nicht die anderen Dateien in den Ordnern (`meta.yaml`, `SOLUTION.md`, `tests.py`). Darin stehen die Lösungen bzw. die bekannten Fehler.

Ihr könnt sie euch auch direkt hier in der Zelle darunter anzeigen lassen.

Untersucht jedes Beispiel im Paar auf:

1. **Semantik:** Was soll der Code tun, und tut er das auch?
2. **Geschätzte Code-Komplexität:** Wie schwer ist der Code zu verstehen – eher einfach, mittel oder schwer?
3. **Fehler:** Fallen euch direkt schon Fehler auf?
4. **Eure eigene Bewertung:** Wie würdet ihr den Code beim Überfliegen bewerten (z. B. als Schulnote)?


In [ ]:
#@title 🔍 Code ansehen
code_auswahl = snippet_auswahl()
code_ausgabe = widgets.Output()


def code_zeigen(*_):
    code_ausgabe.clear_output()
    with code_ausgabe:
        display(HTML(snippet_html(code_auswahl.value)))


code_auswahl.observe(code_zeigen, names="value")
code_zeigen()
display(code_auswahl, code_ausgabe)

## Runde 1 – Frei ausprobieren: Was gehört in einen Judge-Prompt?

Habt ihr schon mal selber einen Judge-Prompt, bzw. eine Review-Anweisung an ein LLM verfasst? Was sollte in solchen Prompts allgemein alles drinstehen, um möglichst alle Fehler im Code zu finden? **Verfasst folgend einen möglichen Prompt für einen LLM-Judge.**

Wählt ein Beispiel, klickt auf **Bewerten** und schickt ein Ergebnis, das ihr zeigen wollt, mit **Ans Dashboard senden** an den Beamer.

- 🎯 **Judge-Score:** Wie gut war der Judge? 2 Punkte für jeden echten Fehler, den er findet, −1 für jeden Fehlalarm.
- 📝 **Code-Score:** Welche Note gibt der Judge dem Code (0–10)?

In [ ]:
#@title ✍️ Runde 1: Euren Prompt bewerten
r1_prompt = prompt_feld("Schreibe einen Judge-Prompt …", hoehe="15em")
r1_snippet = snippet_auswahl()
r1_ausgabe = widgets.Output()
r1_ergebnis = {}


def r1_bewerten():
    r1_ergebnis["letztes"] = bewerte(r1_prompt.value, snippet=r1_snippet.value)
    display(HTML(ergebnis_html(r1_ergebnis["letztes"])))
    r1_senden.disabled = bool(r1_ergebnis["letztes"]["fehler"])


r1_bewerten_knopf = knopf("Bewerten", r1_ausgabe, r1_bewerten)
r1_senden = knopf("Ans Dashboard senden", r1_ausgabe, lambda: sende(r1_ergebnis["letztes"], runde=1),
                  stil="success", aktiv=False)
display(r1_prompt, widgets.HBox([r1_snippet, r1_bewerten_knopf, r1_senden]), r1_ausgabe)
ohne_rechtschreibpruefung()

## Runde 2 – Rollen vergleichen

Im Paper (Chou 2026) bewerten LLMs Code in verschiedenen Rollen – aber niemand prüft, **ob sie recht haben**. Wir können das: Unsere Code-Beispiele enthalten **bekannte Fehler**. Damit messen wir, wie viele davon ein Judge findet und wie oft er Fehlalarm schlägt.

**So haben es die Autor:innen gemacht:** Im Paper bekam das Modell eine von drei Rollen. Die Original-Prompts sind auf Englisch und für C++-Code geschrieben – interessant ist vor allem der Abschnitt *Role Setting*:

- [Strict – Chief Embedded Software Architect](https://github.com/evabowe/software-quality-engineering/blob/main/notebooks/paper_prompts/strict_prompt.md)
- [Objective – Senior System Engineer](https://github.com/evabowe/software-quality-engineering/blob/main/notebooks/paper_prompts/objective_prompt.md)
- [Lenient – Technical Product Manager / Rapid Prototype Developer](https://github.com/evabowe/software-quality-engineering/blob/main/notebooks/paper_prompts/lenient_prompt.md)
- [Quelle und Lizenz](https://github.com/evabowe/software-quality-engineering/blob/main/notebooks/paper_prompts/SOURCE.md)

In Colab findet ihr sie auch links unter 📁 Dateien → `software-quality-engineering/notebooks/paper_prompts/`.

**Eure Aufgabe:** Formuliert selbst zwei verschiedene **Rollen** für euren Judge (z. B. eine strenge und eine wohlwollende), bewertet beide auf demselben Beispiel und sendet sie ans Dashboard. Die Kachel zeigt danach Rolle B.

**Leitfragen (im Paar besprechen)**

1. Wie unterscheidet sich der **Code-Score** zwischen euren Rollen?
2. Ändert die Rolle auch, **wie viele echte Fehler gefunden** werden – oder nur die Note?

In [ ]:
#@title 🎭 Runde 2: Zwei Rollen bewerten
r2_rolle_a = prompt_feld("Rolle A: Beschreibt eine Rolle für den Judge …")
r2_rolle_b = prompt_feld("Rolle B: Beschreibt eine andere Rolle …")
r2_snippet = snippet_auswahl()
r2_ausgabe = widgets.Output()
r2_ergebnis = {}


def r2_bewerten():
    r2_ergebnis["a"] = bewerte(r2_rolle_a.value, snippet=r2_snippet.value)
    r2_ergebnis["b"] = bewerte(r2_rolle_b.value, snippet=r2_snippet.value)
    display(widgets.HBox([widgets.HTML(ergebnis_html(r2_ergebnis["a"], "Rolle A"), layout=widgets.Layout(width="50%")),
                          widgets.HTML(ergebnis_html(r2_ergebnis["b"], "Rolle B"), layout=widgets.Layout(width="50%"))]))
    r2_senden.disabled = bool(r2_ergebnis["a"]["fehler"] and r2_ergebnis["b"]["fehler"])


def r2_senden_beide():
    # Jede Rolle einzeln senden; die Kachel am Dashboard zeigt die zuletzt gesendete (Rolle B).
    sende(r2_ergebnis["a"], runde=2)
    sende(r2_ergebnis["b"], runde=2)


r2_bewerten_knopf = knopf("Beide Rollen bewerten", r2_ausgabe, r2_bewerten)
r2_senden = knopf("Ans Dashboard senden", r2_ausgabe, r2_senden_beide, stil="success", aktiv=False)
display(widgets.HBox([widgets.VBox([r2_rolle_a], layout=widgets.Layout(width="50%")),
                      widgets.VBox([r2_rolle_b], layout=widgets.Layout(width="50%"))]),
        widgets.HBox([r2_snippet, r2_bewerten_knopf, r2_senden]), r2_ausgabe)
ohne_rechtschreibpruefung()

## Runde 3 – Komplexität vergleichen

Jetzt läuft euer Prompt über alle fünf Beispiele mit Komplexitätsstufe, von **L1** (einfach) bis **L5** (schwer).

**Leitfragen (im Paar besprechen)**

1. Findet der Judge bei komplexerem Code **weniger echte Fehler**?
2. Passt die Stufe zu eurer eigenen Einschätzung aus Schritt 0?

In [ ]:
#@title 📈 Runde 3: Alle Stufen L1–L5 bewerten
r3_prompt = prompt_feld("Schreibt euren Prompt für alle fünf Stufen …")
r3_ausgabe = widgets.Output()
r3_ergebnis = {}


def r3_bewerten():
    r3_ergebnis["liste"] = bewerte_alle(r3_prompt.value)
    display(ergebnis_tabelle(r3_ergebnis["liste"]))
    r3_senden.disabled = all(e["fehler"] for e in r3_ergebnis["liste"])


r3_bewerten_knopf = knopf("Alle Stufen bewerten", r3_ausgabe, r3_bewerten)
r3_senden = knopf("Ans Dashboard senden", r3_ausgabe, lambda: sende(r3_ergebnis["liste"], runde=3),
                  stil="success", aktiv=False)
display(r3_prompt, widgets.HBox([r3_bewerten_knopf, r3_senden]), r3_ausgabe)
ohne_rechtschreibpruefung()

## Runde 4 – AI Challenge

**Auftrag:** Findet den besten Prompt, um exakt die vorhandenen Fehler zu finden – **möglichst viele echte Fehler bei möglichst wenigen Fehlalarmen** (2 Punkte pro Treffer, −1 pro Fehlalarm). Am Dashboard wird die beste Gruppe hervorgehoben.

**Regeln**

- **Probeläufe** sind unbegrenzt, jeweils auf einem Beispiel. Sie zählen nicht und werden nicht gesendet.
- **Offizielle Einreichungen** gibt es **drei**. Sie laufen immer über alle fünf Stufen L1–L5 und gehen ans Dashboard – euer Prompt muss also allgemein funktionieren. **Es zählt eure letzte Einreichung** – nicht die beste.

**Tipps**

1. **Rolle:** Gebt dem Judge eine Rolle, die *gründlich* statt *streng* ist.
2. **Prüfschritte:** Lasst ihn systematisch vorgehen, z. B. jede Bedingung, jede Schleife und jeden Rückgabewert prüfen.
3. **Randfälle:** Leere Eingaben, Duplikate, Grenzwerte, Terminierung.
4. **Spezifikation zuerst:** Erst lesen, was der Code laut Docstring tun soll, dann den Code dagegen prüfen.
5. **Nur sichere Befunde:** Fehlalarme kosten Punkte – lieber weniger, dafür begründete Befunde.

In [ ]:
#@title 🏆 Runde 4: Probelauf und Einreichung
MAX_EINREICHUNGEN = 3
r4_prompt = prompt_feld("Euer Prompt für die Challenge …", hoehe="18em")
r4_probe_snippet = snippet_auswahl("Probe auf:")
r4_ausgabe = widgets.Output()
r4_stand = {"einreichungen": 0}


def r4_uebrig():
    return r4_stand["einreichungen"] < MAX_EINREICHUNGEN


def r4_probelauf():
    print("Probelauf – zählt nicht und wird nicht gesendet.")
    display(HTML(ergebnis_html(bewerte(r4_prompt.value, snippet=r4_probe_snippet.value))))


def r4_einreichen():
    if not r4_uebrig():
        print(f"❌ Ihr habt keine Einreichung mehr übrig ({MAX_EINREICHUNGEN} von {MAX_EINREICHUNGEN}).")
        return
    if not r4_prompt.value.strip():
        print("❌ Der Prompt ist leer.")
        return
    liste = bewerte_alle(r4_prompt.value)
    ok = [e for e in liste if not e["fehler"]]
    tabelle = ergebnis_tabelle(liste)
    if not ok:
        display(tabelle)
        print("⚠️ Keine Bewertung hat geklappt – diese Einreichung zählt nicht. Bitte noch einmal einreichen.")
        return
    r4_stand["einreichungen"] += 1
    tabelle.loc["Summe"] = [
        "",
        f"{sum(e['judge_score'] for e in ok)} / {sum(e['judge_max'] for e in ok)}",
        f"Ø {sum(e['code_score'] for e in ok) / len(ok):.1f}",
        f"{sum(e['gefunden'] for e in ok)} / {sum(e['fehler_gesamt'] for e in ok)}",
        sum(e["fehlalarme"] for e in ok),
        "",
    ]
    print(f"Einreichung {r4_stand['einreichungen']} von {MAX_EINREICHUNGEN}:")
    display(tabelle)
    sende(liste, runde=4)
    r4_einreichen_knopf.description = r4_label()


def r4_label():
    if not r4_uebrig():
        return f"Keine Einreichung mehr ({MAX_EINREICHUNGEN}/{MAX_EINREICHUNGEN})"
    return f"Einreichen ({r4_stand['einreichungen'] + 1}/{MAX_EINREICHUNGEN})"


r4_probe_knopf = knopf("Probelauf", r4_ausgabe, r4_probelauf)
r4_einreichen_knopf = knopf(r4_label(), r4_ausgabe, r4_einreichen, stil="warning", aktiv_danach=r4_uebrig)
display(r4_prompt, widgets.HBox([r4_probe_snippet, r4_probe_knopf]), r4_einreichen_knopf, r4_ausgabe)
ohne_rechtschreibpruefung()


## Abschluss

Danke fürs Mitmachen! Bitte beantwortet zum Schluss noch einmal die Umfrage vom Anfang (Link auf der Folie):

*„Würdet ihr einem LLM das Code-Review überlassen – würdet ihr einen LLM-Judge verwenden?“*

Hat sich eure Einschätzung durch den Vortrag und das Hands-on verändert? Wenn ja, wodurch?